In [ ]:
from pathlib import Path
from dtale import show
import polars as pl
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / ".." / "data"
df = pl.read_parquet(
    data_dir / "conversations_clusters" / "cm0i27jdj0000aqpa73ghpcxf.snappy"
)



In [ ]:
df = df.sort("datetime_questions").group_by("questions_cluster_label").agg(
    pl.col("datetime_questions")
    .str.concat("\n NEW CONVERSATION: \n")
    .alias("datetime_questions")
)

d = show(df.to_pandas())
d.open_browser()  


In [ ]:
print(
    df.filter(pl.col("questions_cluster_label") == 256)
    .select(pl.col("datetime_questions"))
    .get_column("datetime_questions")[0]
)


In [ ]:
import numpy as xp
from sklearn.cluster import AgglomerativeClustering
from hdbscan import HDBSCAN
from umap import UMAP
import polars as pl
from pathlib import Path

data_dir = Path().absolute() / ".." / ".." / "data"

df = pl.read_parquet(data_dir / "conversation_embeddings" / "cm0i27jdj0000aqpa73ghpcxf.snappy").filter(pl.col("summary_embedding").is_not_null())

data_dir = Path().absolute() / ".." / ".." / "data"

# Convert the embeddings column to a 2D numpy array
summaries_embeddings = xp.stack(df["summary_embedding"].to_list())

# Reduce the embeddings dimensions
umap_model = UMAP(n_neighbors=15, n_components=100, min_dist=0.1, metric="cosine")
reduced_data = umap_model.fit_transform(summaries_embeddings)

# Move data to cpu if on gpu
reduced_data = reduced_data.astype(xp.float64)

# Clustering for single interests
clusterer = HDBSCAN(
    min_cluster_size=2,
    # gen_min_span_tree=True,
    metric="euclidean",
    prediction_data=True,
).fit(reduced_data)


In [ ]:
from hdbscan.prediction import all_points_membership_vectors
import numpy as np

memebership_vectors = all_points_membership_vectors(clusterer)

# For each vector, get the index of the max value
soft_cluster_labels = memebership_vectors.argmax(axis=1)

len(np.unique(soft_cluster_labels))

